# [실습] 비텍스트 RAG — 표 추출 · 청킹 · 검색분리 (1~4단계)

> PDF 속 **표**는 단순 추출하면 행·열이 깨진다.
> 표를 **살려서 꺼내고 → 청킹하고 → 검색 가능하게** 담는 기초를 직접 코드로 완성해 봅니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 왜 단순 추출이 표를 깨뜨리나 (코드 없음) |
| 2️⃣ 기본 | 샘플 PDF 만들기 · 단순추출 vs 구조보존 |
| 3️⃣ 응용 | ① 작은 표 1청크 ② 큰 표 머리글 복제 |
| 4️⃣ 데이터 활용 | 검색/원본 분리 · 단순 vs 구조보존 비교 · 검수 |

### ✍️ 실습 방법
- 코드 속 `____` (밑줄 4개)를 알맞은 코드로 채운 뒤 실행하세요.
- `# TODO` 주석이 채울 곳을 설명합니다. 막히면 **💡 힌트**를 펼쳐 보세요.
- 각 셀 아래 **✅ 기대 출력**과 내 결과를 비교하세요.
- 셀은 **위에서부터 순서대로** 실행해야 합니다 (앞 셀의 변수를 뒤에서 사용).

# 1️⃣ 단계 — 이론

## PDF엔 "구조 정보"가 없다

```
 PDF의 실체 = "글자 + 그 글자의 (x, y) 좌표"의 모음
   → "여기가 표다 / 문단이다" 라는 구조 정보가 없음
```

## 단순 추출이 깨뜨리는 것

```
 표가 의미를 담는 방식 = 셀의 위치(행, 열)
   | 구분 | 매출 | 증감 |      ← (해외) 와 (-3%) 가
   | 해외 | 85억 | -3% |        '같은 행'이라 연결됨

 단순 추출기는 좌표를 무시하고 읽는 순서대로 일렬로 뽑음
   "구분 매출 증감 해외 85억 -3% ..."
     └─ 어느 값이 어느 항목? 2차원 → 1차원 붕괴 ❌
```

## 해결 — 3단계로 다진다

```
 ① 구조 보존 추출  좌표·선을 분석해 행·열을 복원
 ② 표 청킹         작은 표=통째 1청크 / 큰 표=머리글 복제
 ③ 검색·답변 분리  검색=요약(정확) / 답변=원본(완전)
                   + 추출 결과는 반드시 검수
```

### 🤔 생각해 보기 (코드 없음)
아래 질문에 대한 내 생각을 적어 보세요.

1. PDF에서 표가 "깨진다"는 것은 정확히 무엇이 사라진다는 뜻일까요?
2. 표를 여러 청크로 나눌 때 머리글을 빠뜨리면 어떤 문제가 생길까요?

> ✏️ 답:
> 1.
> 2.

✏️답:  
1. 표가 깨진 다는 것은 데이터가 없다는 것.
2. 머리글을 빠뜨리면 데이터들이 어떤 의미를 가지는지 알 수 없게 된다.

# 2️⃣ 단계 — 샘플 PDF 만들기 · 단순추출 vs 구조보존

### 셀 1 — 도구 설치 (그대로 실행)

`reportlab`(PDF 생성) · `pypdf`(단순 추출) · `pdfplumber`(구조 보존 추출)

In [1]:
!pip install -q reportlab pypdf pdfplumber

# ── Pillow 설치 점검 ───────────────────────────────────────────────
# Pillow가 섞여 설치되면 ImportError: cannot import name '_Ink' 가 납니다.
# 아래에서 자동으로 감지해 깨끗이 재설치합니다.
import subprocess, sys

try:
    import PIL.ImageDraw
    import PIL
    print(f"✅ Pillow {PIL.__version__} 정상")
except ImportError as e:
    print(f"⚠️ Pillow 설치가 깨져 있습니다: {e}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-cache-dir", "pillow"], check=True)
    print("🔧 Pillow 재설치 완료 → 커널(세션) 재시작 후 이어서 실행하세요")

✅ Pillow 12.3.0 정상



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 셀 2 — 표가 든 샘플 PDF 만들기

다음 셀들의 재료로 쓸, **표 1개가 든 PDF**를 코드로 만듭니다.

**TODO**
1. 한글 폰트 `"HYSMyeongJo-Medium"` 을 등록하세요.
2. 표 전체에 두께 `0.5`, 검정색 **격자선(GRID)** 을 넣으세요.
3. 문서에 제목 → 간격 → 캡션 → 간격 → 표 순으로 배치하세요.

<details><summary>💡 힌트</summary>

- 폰트 등록: `pdfmetrics.registerFont(UnicodeCIDFont("폰트이름"))`
- 셀 범위: `(0, 0)` ~ `(-1, -1)` = 표 전체 (좌표는 `(열, 행)`, `-1`은 마지막)
- 격자선: `("GRID", 시작셀, 끝셀, 두께, 색상)`
- 간격: `Spacer(가로, 세로)`
</details>

In [3]:
table_data = [
    ["구분", "매출", "증감"],   # 머리글 행
    ["국내", "120억", "+8%"],
    ["해외", "85억",  "-3%"],
    ["온라인", "47억", "+15%"],
]

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

FONT = "HYSMyeongJo-Medium"

# TODO 1: 한글 폰트 등록
pdfmetrics.registerFont(UnicodeCIDFont(FONT))

styles = getSampleStyleSheet()
for name in ("Title", "Normal"):
    styles[name].fontName = FONT          # 제목·본문 스타일도 한글 폰트로

pdf_table = Table(table_data)
pdf_table.setStyle(TableStyle([
    # TODO 2: 표 전체에 0.5 두께 검정 격자선
    ("GRID", (0,0),(-1,-1), 0.5, colors.black),
    ("FONTNAME", (0, 0), (-1, -1), FONT),
]))

# TODO 3: 제목 → 12pt 간격 → 캡션 → 6pt 간격 → 표
SimpleDocTemplate("sample_report.pdf", pagesize=A4).build([
    Paragraph("2026년 1분기 실적 보고", styles["Title"]),
    Spacer(1, 12),
    Paragraph("[표 1] 부문별 매출 현황", styles["Normal"]),   # ← 표 캡션
    Spacer(1, 6),
    pdf_table,
])
print("✅ sample_report.pdf 생성 완료")

✅ sample_report.pdf 생성 완료


**✅ 기대 출력**: `✅ sample_report.pdf 생성 완료`

> 📌 격자선을 넣는 이유: 추출기가 '선'으로 표를 인식할 수 있게 하기 위함입니다.
> 생성된 PDF를 직접 열어 한글이 깨지지 않았는지 확인해 보세요.

### 셀 3 — 단순 텍스트 추출 (표가 깨지는 쪽)

`pypdf` 로 글자만 뽑습니다. 좌표를 무시하므로 표가 한 줄로 뭉개집니다.

**TODO**: `sample_report.pdf` 를 열고, **첫 페이지**의 텍스트를 추출해 `raw_text` 에 담으세요.

<details><summary>💡 힌트</summary>

- `pypdf.PdfReader("파일명")`
- 첫 페이지: `reader.pages[0]`
- 텍스트 추출 메서드: `.extract_text()`
</details>

In [11]:
import pypdf

# TODO: PDF 열기
reader = pypdf.PdfReader("sample_report.pdf")

# TODO: 첫 페이지 텍스트 추출
raw_text = reader.pages[0].extract_text()

print("───── 단순 추출 결과 ─────")
print(raw_text)
print("───── 내가 궁금한 결과 ─────")
print(reader,"\n")
print(reader.pages,"\n")
print(reader.pages[0].extract_text()[:17])

───── 단순 추출 결과 ─────
2026년 1분기 실적 보고
[표 1] 부문별 매출 현황
구분
매출
증감
국내
120억
+8%
해외
85억
-3%
온라인
47억
+15%

───── 내가 궁금한 결과 ─────

[PageObject(0)] 

2026년 1분기 실적 보고
[


**✅ 기대 출력** (값이 한 줄씩 흩어짐)
```
2026년 1분기 실적 보고
[표 1] 부문별 매출 현황
구분
매출
증감
국내
120억
...
```
> ⚠️ "구분 / 매출 / 증감"이 어느 셀과 짝인지 사라졌습니다 → **행·열 붕괴**

### 셀 4 — 구조 보존 추출 (표가 살아나는 쪽)

`pdfplumber` 는 글자의 **좌표·선을 분석**해 *"같은 y = 같은 행"* 으로 표 구조를 복원합니다.

**TODO**: 첫 페이지에서 표 목록을 추출하고, 그중 **첫 번째 표**를 `extracted_table` 에 담으세요.

<details><summary>💡 힌트</summary>

- 표 목록 추출: `page.extract_tables()` → 표들의 리스트 (각 표는 2차원 리스트)
- 첫 번째 표: `[0]`
</details>

In [7]:
import pdfplumber

with pdfplumber.open("sample_report.pdf") as pdf:
    # TODO: 첫 페이지의 첫 번째 표 (2차원 리스트)
    extracted_table = pdf.pages[0].extract_tables()[0]

for row in extracted_table:
    print(row)

['구분', '매출', '증감']
['국내', '120억', '+8%']
['해외', '85억', '-3%']
['온라인', '47억', '+15%']


**✅ 기대 출력**
```
['구분', '매출', '증감']
['국내', '120억', '+8%']
['해외', '85억', '-3%']
['온라인', '47억', '+15%']
```
> 💡 셀 3과 비교하세요. `['해외', '85억', '-3%']` 처럼 한 행이 묶여 **관계가 살아 있습니다.**

| 도구 | 강점 | 비용 |
|------|------|------|
| LlamaParse | 복잡 표 → 마크다운 품질 최상 | 페이지당 과금 |
| Unstructured | 형식 다양성(PDF·PPT·docx) | 오픈소스 무료 |
| Docling | 표 구조·읽기순서 보존 특화 | 오픈소스 무료 |
| pdfplumber | 가볍고 빠른 표 추출(실습용) | 오픈소스 무료 |

# 3️⃣ 단계 — 표를 RAG에 담는 형태로 만들기

## ① 작은 표 → 마크다운 1청크

### 셀 5 — 표를 마크다운으로 변환 (LLM이 잘 읽는 형식)

작은 표(~20행)는 **표 1개 = 1청크**. 행 도중에 쪼개지 않습니다.
+ **캡션을 맥락 한 줄로** 붙여 "무엇에 대한 표인지" 알 수 있게 합니다.

**TODO**
1. `table_to_markdown()` 의 구분선(`| --- | --- | --- |`)과 본문 행을 완성하세요.
2. 캡션과 마크다운 표를 **줄바꿈**으로 이어 `table_chunk` 를 만드세요.

<details><summary>💡 힌트</summary>

- 열 개수만큼 `"---"` 반복: `["---"] * len(rows[0])`
- 본문: 머리글 다음 행부터 → `rows[1:]`, 각 행은 머리글과 같은 방식으로 `" | ".join(r)`
- 줄바꿈 연결: `caption + "\n" + ...`
</details>

In [ ]:
def table_to_markdown(rows):
    header  = "| " + " | ".join(rows[0]) + " |"
    # TODO 1-a: 열 개수만큼 '---' 를 넣은 구분선
    divider = "| " + " | ".join(____) + " |"
    # TODO 1-b: 머리글을 제외한 나머지 행들을 마크다운 행으로
    body    = ["| " + " | ".join(r) + " |" for r in ____]
    return "\n".join([header, divider] + body)

caption = "[표 1] 2026년 1분기 부문별 매출 현황"

# TODO 2: 캡션 + 줄바꿈 + 마크다운 표
table_chunk = ____
print(table_chunk)

**✅ 기대 출력**
```
[표 1] 2026년 1분기 부문별 매출 현황
| 구분 | 매출 | 증감 |
| --- | --- | --- |
| 국내 | 120억 | +8% |
| 해외 | 85억 | -3% |
| 온라인 | 47억 | +15% |
```
> 💡 마크다운 표는 한 행이 한 줄에 완결돼 LLM이 행·열 관계를 정확히 읽습니다.

## ② 큰 표 → 머리글 복제하며 분할

### 셀 6 — 큰 표는 조각마다 머리글 복제

100행짜리 큰 표는 통째 1청크 불가. 행 그룹으로 나누되 —
**매 조각에 머리글 행을 복제**해 붙여야 열 의미가 살아남습니다.

**TODO**
1. `big_table` 을 머리글(`header`)과 나머지 행(`rows`)으로 나누세요.
2. `chunk_size` 행씩 잘라 낸 `block` 앞에 **머리글을 붙이세요.**

<details><summary>💡 힌트</summary>

- 첫 행: `big_table[0]`, 나머지: `big_table[1:]`
- `range(시작, 끝, 간격)` 으로 `chunk_size` 씩 건너뛰기
- 머리글 복제: `[header] + block`
</details>

In [ ]:
big_table = [
    ["구분", "매출", "증감"],
    ["국내", "120억", "+8%"], ["해외", "85억", "-3%"],
    ["온라인", "47억", "+15%"], ["오프라인", "73억", "-1%"],
    ["B2B", "210억", "+22%"], ["B2C", "88억", "+5%"],
]

# TODO 1: 머리글 / 나머지 행 분리
header, rows = ____, ____

chunk_size = 2
chunks = []
for i in range(0, len(rows), ____):
    block = rows[i:i + chunk_size]
    # TODO 2: 머리글 복제 + 현재 블록
    chunk = ____
    chunks.append(table_to_markdown(chunk))

for n, chunk in enumerate(chunks, 1):
    print(f"───── 조각 {n} ─────")
    print(chunk, "\n")

**✅ 기대 출력**: 조각 3개, **모든 조각의 첫 줄이** `| 구분 | 매출 | 증감 |`
```
───── 조각 1 ─────
| 구분 | 매출 | 증감 |
| --- | --- | --- |
| 국내 | 120억 | +8% |
| 해외 | 85억 | -3% |
...
```
> 💡 큰 표에서 가장 흔한 실수인 **머리글 분실**을 막는 규칙입니다.

# 4️⃣ 단계 — 검색/원본 분리 · 비교 · 검수

### 셀 7 — 검색용 요약 / 답변용 원본 분리

표를 마크다운 그대로 임베딩하면 기호·숫자가 많아 의미 매칭이 약합니다.
그래서 **검색은 '요약'으로, 답변은 '원본'으로** 분리합니다.

```
 색인:  표 ─▶ 요약(검색용)  ─▶ search_index
          └─▶ 원본(답변용) ─▶ docstore        같은 id 로 연결 🔑
 검색:  질문 ─▶ 요약으로 검색 ─▶ id ─▶ docstore에서 원본 꺼내 전달
```

**TODO**
1. 고유 `doc_id` 를 만드세요.
2. **같은 `doc_id`** 로 `docstore`(원본)와 `search_index`(요약)에 저장하세요.
3. 찾은 id로 `docstore` 에서 원본을 꺼내 출력하세요.

<details><summary>💡 힌트</summary>

- 고유 id: `str(uuid.uuid4())`
- 딕셔너리: `{키: 값}`
- 원본 꺼내기: `docstore[found_id]`
</details>

In [ ]:
import uuid

table_original = table_chunk                                                      # 답변용 (셀 5 결과)
table_summary  = "2026년 1분기 부문별 매출과 증감률 표. 국내 해외 온라인 매출 실적"  # 검색용

# TODO 1: 고유 id
doc_id = ____

# TODO 2: 같은 id 로 두 저장소에 저장
docstore     = {____: ____}   # 답변용 저장소 (원본)
search_index = {____: ____}   # 검색용 저장소 (요약)

def search(query):
    # 질문 단어 중 하나라도 요약에 들어 있으면 그 id 반환 (단순 키워드 검색)
    for _id, summary in search_index.items():
        if any(word in summary for word in query.split()):
            return _id
    return None

found_id = search("해외 매출 증감 알려줘")
print("🔎 검색은 '요약'으로 → id:", found_id[:8])

# TODO 3: 찾은 id 로 원본 꺼내기
print("📦 LLM엔 '원본' 전달:\n", ____)

**✅ 기대 출력**: id 앞 8자리(실행마다 다름) + 셀 5의 마크다운 표 원본

> 💡 그림·차트도 똑같습니다: **그림 → 요약 텍스트(검색용) + 원본 그림(답변용)**.

### 셀 8 — 단순 vs 구조보존, 같은 질문에 답 비교 (핵심!)

같은 질문을 **단순 추출본**과 **구조 보존본**에 던져 답이 어떻게 갈리는지 봅니다.
(LLM 없이, 규칙 기반으로 차이만 드러냄)

**TODO**: `answer_from_struct()` 에서
1. 표의 첫 행을 머리글(`head`)로 잡고,
2. 머리글을 제외한 행을 돌며 **첫 열(구분)에 키워드가 있는 행**을 찾고,
3. `머리글: 값` 쌍으로 묶어 반환하세요.

<details><summary>💡 힌트</summary>

- 머리글: `extracted_table[0]` / 데이터 행: `extracted_table[1:]`
- 첫 열: `row[0]`
- 짝 맞추기: `zip(head, row)`
</details>

In [ ]:
def answer_from_plain(query):
    # 단순 추출본(1차원 텍스트)에서 질문 첫 단어 주변 30글자를 긁어옴
    key = query.split()[0]
    flat = " ".join(raw_text.split())
    idx = flat.find(key)
    return flat[idx:idx + 30] + " ..." if idx >= 0 else "(못 찾음)"

def answer_from_struct(query):
    # 구조 보존본(2차원 표)에서 해당 '행'을 정확히 찾음
    key = query.split()[0]
    # TODO 1: 머리글 행
    head = ____
    # TODO 2: 데이터 행 순회
    for row in ____:
        if key in ____:
            # TODO 3: '머리글: 값' 쌍으로 연결
            return " / ".join(f"{h}: {v}" for h, v in ____)
    return "(못 찾음)"

q = "해외 매출 증감"
print(f"🧑 질문: {q}\n")
print("─── 단순 추출 답 ───")
print(answer_from_plain(q))
print("\n─── 구조 보존 답 ───")
print(answer_from_struct(q))

**✅ 기대 출력**
```
─── 단순 추출 답 ───
해외 85억 -3% 온라인 47억 +15% ...

─── 구조 보존 답 ───
구분: 해외 / 매출: 85억 / 증감: -3%
```
> ✅ **같은 문서인데 추출 방식만 달라** 답 품질이 갈립니다.

### 셀 9 — 추출 결과 검수

추출은 한 번에 완벽하지 않습니다. **병합 셀**은 빈칸/`None`으로 깨지기 쉬우니,
추출 직후 빈 셀이 있는지 점검하는 습관을 들입니다.

**TODO**: 모든 셀을 돌며 값이 `None` 이거나 빈 문자열 `""` 인 셀의 `(행, 열)` 위치를 `bad` 에 모으세요.

<details><summary>💡 힌트</summary>

- 이중 반복 리스트 컴프리헨션: `[(r, c) for r, row in enumerate(rows) for c, cell in enumerate(row) if 조건]`
- 조건: `cell in (None, "")`
</details>

In [ ]:
def validate_table(rows):
    # TODO: 빈/None 셀의 (행, 열) 좌표 목록
    bad = ____
    return "⚠️ 빈/병합 의심 셀: " + str(bad) if bad else "✅ 이상 없음"

print(validate_table(extracted_table))                 # 정상 표
print(validate_table([["연도", None, None],             # 병합 흉내 표
                       ["2026", "120억", "+8%"]]))

**✅ 기대 출력**
```
✅ 이상 없음
⚠️ 빈/병합 의심 셀: [(0, 1), (0, 2)]
```
> 💡 **"추출하면 끝"이 아니라 "추출 → 검수"** 가 기초 습관입니다.

---
# 🎓 정리 — 적용한 이론

| 이론 | 이 실습에서 한 일 |
|------|-------------------|
| PDF엔 구조 정보 없음 | 단순 추출로 행·열 붕괴를 직접 확인 |
| 구조 보존 추출 | `extract_tables()` 로 행·열 복원 |
| 작은 표 청킹 | 표 1개 = 1청크 + 캡션을 맥락 한 줄로 |
| 큰 표 청킹 | 조각마다 머리글 행 복제 |
| 검색/답변 분리 | `id`로 연결, 검색=요약·전달=원본 |
| 추출 검수 | 빈/병합 셀 점검 습관 |

## 🤔 더 해보기 (선택)

1. 셀 2의 `table_data` 를 본인이 만든 표(예: 제품 스펙 표)로 바꾼 뒤 셀 3~5를 다시 실행해 보세요.
2. 셀 6의 `chunk_size` 를 `3` 으로 바꾸면 조각이 어떻게 나뉘는지 확인해 보세요.
3. 셀 8의 `q` 를 `"국내 매출 증감"` 등으로 바꿔도 구조 보존 쪽이 정확한 행을 반환하는지 살펴보세요.
4. 표 구조를 완벽히 살렸는데도 *"모델 ABC-1234의 소비전력"* 같은 **정확한 품번 검색**은 왜 빗나갈 수 있을까요?

> ✏️ 내 생각: